In [4]:
import numpy as np

from embedder import Embedder

from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents
from minsearch import Index, VectorSearch

In [5]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [6]:
embedd = Embedder()

## Q1

In [7]:
res1 = embedd.encode('How does approximate nearest neighbor search work?')
print(f'Result: {res1[0]:.3f}')

Result: -0.021


## Q2

In [8]:
for page in documents:
  if page.get('filename') == '02-vector-search/lessons/07-sqlitesearch-vector.md':
    text = page.get('content')
res2 = embedd.encode(text)

In [9]:
dot_product = np.dot(res1, res2)
norm_q = np.linalg.norm(res1)
norm_d = np.linalg.norm(res2)

cosine_similarity = dot_product / (norm_q * norm_d)
print(f"Cosine Similarity: {cosine_similarity:.4f}")

Cosine Similarity: 0.3611


## Q3

In [10]:
chunks = chunk_documents(documents, size=2000, step=1000)

In [11]:
X = []
for chunk in chunks:
  X.append(embedd.encode(chunk['content']))
X = np.array(X)

In [12]:
scores = X.dot(res1)

In [13]:
idx = scores.argmax()
chunks[idx]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

## Q4

In [14]:
query1 = embedd.encode('What metric do we use to evaluate a search engine?')

In [15]:
vindex = VectorSearch()
vindex.fit(X, chunks)

In [16]:
results1 = vindex.search(query1, num_results=5)

In [17]:
for result in results1:
  print(f'{result['filename']}')

04-evaluation/lessons/05-search-metrics.md
04-evaluation/lessons/01-intro.md
01-agentic-rag/lessons/05-search.md
04-evaluation/lessons/01-intro.md
04-evaluation/lessons/15-next-steps.md


## Q5

In [18]:
index = Index(
    text_fields=["content"]
)
index.fit(chunks)

In [19]:
query2 = 'How do I store vectors in PostgreSQL?'

In [20]:
results1 = vindex.search( embedd.encode(query2), num_results=5)

In [21]:
results2 = index.search(query2, num_results=5)

In [22]:
result1 = [result['filename'] for result in results1]
result2 = [result['filename'] for result in results2]

In [30]:
list(set(result1) - set(result2))

['02-vector-search/lessons/08-pgvector.md']

## Q6

In [31]:
q3 = 'How do I give the model access to tools?'

In [32]:
vector_results = vindex.search( embedd.encode(q3), num_results=5)
text_results = index.search(q3, num_results=5)

In [33]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [34]:
results = rrf([vector_results, text_results])

In [35]:
results[0]['filename']

'01-agentic-rag/lessons/13-function-calling.md'